# Multi-Lesion DR Staged Learning Pipeline

## Before you start -- Kaggle setup checklist

1. **GPU**: Click the **Settings** tab on the right panel -> **Accelerator** -> select **GPU T4 x2** (or P100)
2. **Internet**: Click **Settings** -> **Internet** -> toggle **On**
3. **Add these datasets** (click **+ Add Data** on the right panel, search by name):
   - Search: `diabetic-retinopathy-detection` (Competition) -- this is EyePACS
   - Search: `ddrdataset` (Dataset by mariaherrerot) -- this is DDR
   - Search: `indian-diabetic-retinopathy-image-dataset` (Dataset) -- this is IDRiD

Once you have done those 3 steps, run every cell from top to bottom with **Shift+Enter**.

---
## STEP 1: Clone the code and install dependencies

In [ ]:
%%bash
# Clone the repository
cd /kaggle/working
if [ ! -d "repo" ]; then
    git clone --branch feature/dr-multi-lesion-staged-pipeline \
        https://github.com/Ziad-Ahmed-Zaska/ECG-Image-Classification-Using-CNN.git repo
fi
echo "Done cloning."

In [ ]:
# Install missing packages (most are pre-installed on Kaggle)
!pip install -q opencv-python-headless scikit-learn

In [ ]:
# Make the pipeline importable
import sys
sys.path.insert(0, '/kaggle/working/repo')

# Verify
import dr_pipeline
print(f'Pipeline version: {dr_pipeline.__version__}')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## STEP 2: Configure all paths

Kaggle mounts datasets at `/kaggle/input/<dataset-slug>/`. The cell below auto-detects which datasets you added.

In [ ]:
import os
from pathlib import Path

# ============================================================
# AUTO-DETECT dataset paths
# If a path doesn't exist, that stage will be skipped gracefully
# ============================================================

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/dr_outputs')
OUTPUT.mkdir(exist_ok=True)

# EyePACS (from the Diabetic Retinopathy Detection competition)
EYEPACS_ROOT = None
for candidate in ['diabetic-retinopathy-detection', 'diabetic-retinopathy-resized']:
    p = INPUT / candidate
    if p.is_dir():
        EYEPACS_ROOT = p
        break

# DDR
DDR_ROOT = None
for candidate in ['ddrdataset', 'ddr-dataset', 'ddr']:
    p = INPUT / candidate
    if p.is_dir():
        DDR_ROOT = p
        break

# IDRiD
IDRID_ROOT = None
for candidate in ['indian-diabetic-retinopathy-image-dataset', 'idrid-dataset', 'idrid']:
    p = INPUT / candidate
    if p.is_dir():
        IDRID_ROOT = p
        break

print('=== Dataset Detection ===')
for name, path in [('EyePACS', EYEPACS_ROOT), ('DDR', DDR_ROOT), ('IDRiD', IDRID_ROOT)]:
    if path and path.is_dir():
        print(f'  [OK]  {name}: {path}')
        # Show top-level contents
        print(f'        Contents: {os.listdir(path)[:10]}')
    else:
        print(f'  [--]  {name}: NOT FOUND -- add it via "+ Add Data" on the right panel')

# Override pipeline output dirs to write to /kaggle/working/
import dr_pipeline.config as cfg
cfg.OUTPUT_DIR = OUTPUT
cfg.CHECKPOINT_DIR = OUTPUT / 'checkpoints'
cfg.LOG_DIR = OUTPUT / 'logs'
cfg.FIGURE_DIR = OUTPUT / 'figures'
cfg.ensure_dirs()
print(f'\nOutputs will be saved to: {OUTPUT}')

---
## STEP 3: Build EyePACS metadata (patient-level splits)

The EyePACS competition has a `trainLabels.csv` with columns `image` and `level` (0-4).
We convert this into the pipeline's format and create patient-level train/val/test splits.

In [ ]:
import pandas as pd
from dr_pipeline.datasets.dataset_manifest import DatasetManifest

eyepacs_manifest = None
eyepacs_images_dir = None

if EYEPACS_ROOT is not None:
    # Find the labels CSV
    labels_csv = None
    for name in ['trainLabels.csv', 'trainLabels.csv.zip']:
        candidate = EYEPACS_ROOT / name
        if candidate.exists():
            labels_csv = candidate
            break
    
    # Find the images directory
    for name in ['train', 'resized_train', 'resized_train_cropped']:
        candidate = EYEPACS_ROOT / name
        if candidate.is_dir():
            eyepacs_images_dir = candidate
            break
    
    if labels_csv and eyepacs_images_dir:
        print(f'Labels CSV: {labels_csv}')
        print(f'Images dir: {eyepacs_images_dir}')
        
        # Read and convert
        df = pd.read_csv(labels_csv)
        print(f'Total images in CSV: {len(df)}')
        
        grade_map = {0: 'no_dr', 1: 'mild', 2: 'moderate', 3: 'severe', 4: 'proliferative'}
        df['label'] = df['level'].map(grade_map)
        
        # Extract patient ID by stripping _left / _right
        df['patient_id'] = df['image'].str.replace(r'_(left|right)$', '', regex=True)
        
        # Find actual image extension
        sample_files = os.listdir(eyepacs_images_dir)[:5]
        ext = '.jpeg'  # default
        if sample_files:
            ext = '.' + sample_files[0].split('.')[-1]
        print(f'Image extension: {ext}')
        
        df['image_path'] = df['image'] + ext
        
        # Only keep images that actually exist on disk
        df['exists'] = df['image_path'].apply(lambda x: (eyepacs_images_dir / x).exists())
        print(f'Images found on disk: {df["exists"].sum()} / {len(df)}')
        df = df[df['exists']].drop(columns=['exists'])
        
        # Save metadata CSV
        meta_path = OUTPUT / 'eyepacs_metadata.csv'
        df[['image_path', 'patient_id', 'label']].to_csv(meta_path, index=False)
        
        # Build patient-level splits
        eyepacs_manifest = DatasetManifest(
            dataset_name='eyepacs',
            root_dir=str(eyepacs_images_dir),
            train_ratio=0.70,
            val_ratio=0.15,
        )
        eyepacs_manifest.load_metadata(str(meta_path))
        eyepacs_manifest.build_patient_splits()
        eyepacs_manifest.save(str(OUTPUT / 'eyepacs_manifest'))
        
        print('\n=== EyePACS Splits ===')
        for s in eyepacs_manifest.summary():
            print(f'  {s.split}: {s.num_patients} patients, {s.num_images} images')
            print(f'    Labels: {s.label_distribution}')
    else:
        print(f'Could not find labels CSV or images dir in {EYEPACS_ROOT}')
        print(f'Contents: {os.listdir(EYEPACS_ROOT)}')
else:
    print('EyePACS not available. Stage 2 will use ImageNet-pretrained weights.')

---
## STEP 4: Stage 2 -- Encoder Pretraining

Choose a training variant:
- **A** = Self-supervised SimCLR (no labels needed, learns anatomy features)
- **B** = Supervised DR grading (faster, uses labels)
- **C** = Both: SSL first, then supervised fine-tune (best quality, but slowest)

If EyePACS is not available, we fall back to ImageNet-pretrained weights.

In [ ]:
# ============================================================
# CONFIGURATION -- change these as needed
# ============================================================
VARIANT = 'B'           # 'A', 'B', or 'C'
BATCH_SIZE = 32         # reduce to 16 or 8 if you hit OOM
ENCODER_NAME = 'resnet50'  # 'resnet18' is lighter
NUM_EPOCHS_SSL = 20     # SimCLR epochs (increase for better features)
NUM_EPOCHS_SUP = 15     # Supervised DR grading epochs
LEARNING_RATE = 1e-4
MAX_IMAGES = None       # Set to e.g. 5000 for a quick test run

print(f'Variant: {VARIANT}, Batch: {BATCH_SIZE}, Encoder: {ENCODER_NAME}')
print(f'SSL epochs: {NUM_EPOCHS_SSL}, Supervised epochs: {NUM_EPOCHS_SUP}')
if MAX_IMAGES:
    print(f'QUICK MODE: using only {MAX_IMAGES} images')

In [ ]:
from dr_pipeline.config import TrainConfig
from dr_pipeline.models.encoder import DREncoder
from dr_pipeline.datasets.eyepacs import EyePACSDataset
from dr_pipeline.training.train_pretrain import train_ssl, train_supervised
from torch.utils.data import DataLoader
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

train_cfg = TrainConfig(
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    epochs=NUM_EPOCHS_SUP,
    ssl_epochs=NUM_EPOCHS_SSL,
    encoder_name=ENCODER_NAME,
    mixed_precision=True,
    num_workers=2,
)

encoder = None

if eyepacs_manifest is not None:
    train_records = eyepacs_manifest.get_split('train')
    val_records = eyepacs_manifest.get_split('val')
    
    if MAX_IMAGES:
        train_records = train_records[:MAX_IMAGES]
        val_records = val_records[:MAX_IMAGES // 5]
        print(f'Quick mode: {len(train_records)} train, {len(val_records)} val')

    # --- Variant A or C: SSL pretraining ---
    if VARIANT in ('A', 'C'):
        print('\n' + '='*60)
        print('Stage 2A: Self-supervised pretraining (SimCLR)')
        print('='*60)
        ssl_ds = EyePACSDataset(train_records, root_dir=str(eyepacs_images_dir), mode='ssl')
        ssl_loader = DataLoader(ssl_ds, batch_size=BATCH_SIZE, shuffle=True,
                                num_workers=2, pin_memory=True, drop_last=True)
        encoder = train_ssl(train_cfg, ssl_loader)
        print('SSL pretraining complete.')

    # --- Variant B or C: Supervised DR grading ---
    if VARIANT in ('B', 'C'):
        print('\n' + '='*60)
        print('Stage 2B: Supervised DR grading')
        print('='*60)
        train_ds = EyePACSDataset(train_records, root_dir=str(eyepacs_images_dir), mode='supervised')
        val_ds = EyePACSDataset(val_records, root_dir=str(eyepacs_images_dir), mode='supervised')
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=2, pin_memory=True)
        encoder = train_supervised(train_cfg, train_loader, val_loader, encoder=encoder)
        print('Supervised DR grading complete.')
else:
    print('No EyePACS data -- using ImageNet-pretrained encoder.')
    encoder = DREncoder(backbone_name=ENCODER_NAME, pretrained_imagenet=True)

print(f'\nEncoder ready. Feature dimension: {encoder.get_feat_dim()}')

---
## STEP 5: Build DDR metadata

In [ ]:
ddr_manifest = None

if DDR_ROOT is not None:
    print(f'DDR root: {DDR_ROOT}')
    print(f'Contents: {os.listdir(DDR_ROOT)}')
    
    # Walk the directory to find all images and any label files
    all_images = []
    for root, dirs, files in os.walk(DDR_ROOT):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                rel_path = os.path.relpath(os.path.join(root, f), DDR_ROOT)
                all_images.append(rel_path)
    
    print(f'Total images found: {len(all_images)}')
    
    if all_images:
        # Build a metadata CSV
        # DDR typically has lesion annotation files; for now we create placeholder labels
        # that you should replace with the actual DDR lesion annotations
        records = []
        for img_path in all_images:
            fname = os.path.basename(img_path).split('.')[0]
            records.append({
                'image_path': img_path,
                'patient_id': fname,  # unique per image if no patient info
                'label': 'unknown',
                'microaneurysm': 0,
                'haemorrhage': 0,
                'hard_exudate': 0,
                'soft_exudate': 0,
            })
        
        df_ddr = pd.DataFrame(records)
        
        # Try to load actual DDR lesion labels if available
        for label_file in DDR_ROOT.rglob('*.txt'):
            print(f'  Found label file: {label_file}')
        for label_file in DDR_ROOT.rglob('*.csv'):
            print(f'  Found CSV file: {label_file}')
        
        ddr_meta_path = OUTPUT / 'ddr_metadata.csv'
        df_ddr.to_csv(ddr_meta_path, index=False)
        
        ddr_manifest = DatasetManifest(
            dataset_name='ddr',
            root_dir=str(DDR_ROOT),
            train_ratio=0.70,
            val_ratio=0.15,
        )
        ddr_manifest.load_metadata(str(ddr_meta_path))
        ddr_manifest.build_patient_splits()
        
        print(f'\nDDR splits:')
        for s in ddr_manifest.summary():
            print(f'  {s.split}: {s.num_images} images')
    else:
        print('No images found in DDR directory.')
else:
    print('DDR dataset not found. Skipping Stage 3.')
    print('To add it: click "+ Add Data" and search for "ddrdataset"')

---
## STEP 6: Stage 3 -- Multi-Lesion Classification on DDR

In [ ]:
from dr_pipeline.datasets.ddr import DDRDataset
from dr_pipeline.training.train_lesion_classifier import train_lesion_classifier

lesion_model = None
thresholds = None

USE_MULTI_SCALE = False  # Set True for the MIL patch branch (slower, better for tiny lesions)
LESION_BATCH_SIZE = 16   # Reduce if OOM
LESION_EPOCHS = 20

if ddr_manifest is not None:
    print('='*60)
    print(f'Stage 3: Multi-lesion classification (multi_scale={USE_MULTI_SCALE})')
    print('='*60)
    
    ddr_train_records = ddr_manifest.get_split('train')
    ddr_val_records = ddr_manifest.get_split('val')
    
    if MAX_IMAGES:
        ddr_train_records = ddr_train_records[:MAX_IMAGES]
        ddr_val_records = ddr_val_records[:MAX_IMAGES // 5]
    
    ddr_train_ds = DDRDataset(ddr_train_records, root_dir=str(DDR_ROOT),
                              return_patches=USE_MULTI_SCALE)
    ddr_val_ds = DDRDataset(ddr_val_records, root_dir=str(DDR_ROOT),
                            return_patches=USE_MULTI_SCALE)
    
    ddr_train_loader = DataLoader(ddr_train_ds, batch_size=LESION_BATCH_SIZE,
                                  shuffle=True, num_workers=2, pin_memory=True)
    ddr_val_loader = DataLoader(ddr_val_ds, batch_size=LESION_BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)
    
    lesion_cfg = TrainConfig(
        batch_size=LESION_BATCH_SIZE,
        learning_rate=1e-4,
        epochs=LESION_EPOCHS,
        encoder_name=ENCODER_NAME,
    )
    
    lesion_model, thresholds = train_lesion_classifier(
        lesion_cfg, ddr_train_loader, ddr_val_loader,
        encoder=encoder, multi_scale=USE_MULTI_SCALE,
    )
    print(f'\nOptimal per-class thresholds: {thresholds.tolist()}')
else:
    print('DDR not available -- skipping Stage 3.')

---
## STEP 7: Stage 4 -- Explainability (Grad-CAM)

In [ ]:
from dr_pipeline.evaluation.explainability import GradCAM, generate_attributions
from dr_pipeline.config import LESION_CLASSES
import matplotlib.pyplot as plt
import numpy as np

if lesion_model is not None:
    print('='*60)
    print('Stage 4: Generating Grad-CAM attribution maps')
    print('='*60)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    lesion_model.to(device).eval()
    
    # Grab a batch for visualization
    for batch in ddr_val_loader:
        if USE_MULTI_SCALE:
            sample_images = batch[0][:4].to(device)
        else:
            sample_images = batch[0][:4].to(device)
        break
    
    print(f'Generating attributions for {sample_images.shape[0]} images...')
    attributions = generate_attributions(lesion_model, sample_images, method='gradcam')
    
    # Visualize
    n_images = sample_images.shape[0]
    fig, axes = plt.subplots(n_images, len(LESION_CLASSES) + 1,
                             figsize=(4 * (len(LESION_CLASSES) + 1), 4 * n_images))
    
    # De-normalize for display
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
    
    for i in range(n_images):
        img = sample_images[i] * std + mean
        img = img.permute(1, 2, 0).cpu().numpy().clip(0, 1)
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        for j, cls_name in enumerate(LESION_CLASSES):
            hm = attributions[cls_name][i]
            axes[i, j+1].imshow(img)
            axes[i, j+1].imshow(hm, cmap='jet', alpha=0.5)
            axes[i, j+1].set_title(cls_name)
            axes[i, j+1].axis('off')
    
    plt.tight_layout()
    fig.savefig(str(OUTPUT / 'figures' / 'attribution_montage.png'), dpi=150)
    plt.show()
    print('Attribution montage saved.')
else:
    print('No lesion model -- skipping Stage 4.')

---
## STEP 8: Stage 5 -- Segmentation on IDRiD

In [ ]:
if IDRID_ROOT is not None:
    print('='*60)
    print('Stage 5: Segmentation setup')
    print('='*60)
    print(f'IDRiD root: {IDRID_ROOT}')
    print(f'Contents: {os.listdir(IDRID_ROOT)}')
    
    # Walk the directory to understand the layout
    for root, dirs, files in os.walk(IDRID_ROOT):
        depth = root.replace(str(IDRID_ROOT), '').count(os.sep)
        if depth < 3:
            indent = ' ' * 2 * depth
            print(f'{indent}{os.path.basename(root)}/')
            if depth == 2:
                print(f'{indent}  ({len(files)} files)')
    
    print('\n--- IDRiD segmentation training is dataset-layout-specific. ---')
    print('The pipeline module is ready at dr_pipeline.training.train_segmentation.')
    print('You need to create an IDRiD metadata CSV mapping images to mask paths.')
    print('See dr_pipeline/README.md for the expected CSV schema.')
else:
    print('IDRiD not found -- skipping Stage 5.')
    print('To add it: click "+ Add Data" and search for "indian-diabetic-retinopathy-image-dataset"')

---
## STEP 9: Evaluation Report

In [ ]:
from dr_pipeline.evaluation.metrics import classification_report, per_lesion_auc
from dr_pipeline.evaluation.calibration import expected_calibration_error
import json

if lesion_model is not None and ddr_manifest is not None:
    print('='*60)
    print('Evaluation on DDR test set')
    print('='*60)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    lesion_model.to(device).eval()
    
    # Build test loader (no patches for evaluation speed)
    ddr_test_ds = DDRDataset(ddr_manifest.get_split('test'), root_dir=str(DDR_ROOT),
                             return_patches=False)
    test_loader = DataLoader(ddr_test_ds, batch_size=16, shuffle=False, num_workers=2)
    
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            logits = lesion_model(images.to(device))
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    
    # Metrics
    report = classification_report(all_labels, all_probs, thresholds.numpy())
    
    print('\n--- ROC-AUC per lesion ---')
    for k, v in report['roc_auc'].items():
        print(f'  {k}: {v:.4f}')
    
    print('\n--- PR-AUC per lesion ---')
    for k, v in report['pr_auc'].items():
        print(f'  {k}: {v:.4f}')
    
    print('\n--- F1 scores ---')
    for k, v in report['f1_scores'].items():
        print(f'  {k}: {v:.4f}')
    
    # Calibration
    ece, _, _, _ = expected_calibration_error(all_labels, all_probs)
    print(f'\nExpected Calibration Error (ECE): {ece:.4f}')
    
    # Save
    report_path = OUTPUT / 'evaluation_report.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2, default=str)
    print(f'\nFull report saved to: {report_path}')
else:
    print('No model or test data available for evaluation.')

---
## STEP 10: Save Checkpoints

Kaggle notebooks can run for up to 12 hours. Save your checkpoints so you can resume later.

In [ ]:
print('Saved outputs:')
for p in sorted(OUTPUT.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'  {p.relative_to(OUTPUT)}  ({size_mb:.1f} MB)')

print(f'\nAll outputs are in: {OUTPUT}')
print('Download them from the Output tab after the notebook finishes.')

---
## Unit Tests (optional sanity check)

In [ ]:
!cd /kaggle/working/repo && python -m pytest tests/test_dr_pipeline.py -v --tb=short 2>&1 | tail -25

---
## Troubleshooting

| Problem | Solution |
|---------|----------|
| `RuntimeError: CUDA out of memory` | Reduce `BATCH_SIZE` to 8 or use `ENCODER_NAME = 'resnet18'` |
| Dataset `NOT FOUND` | Click **+ Add Data** on the right panel and search for the dataset name |
| `No module named dr_pipeline` | Re-run the first cells to clone the repo and set `sys.path` |
| Training is too slow | Set `MAX_IMAGES = 5000` for a quick test, or reduce `NUM_EPOCHS_*` |
| `FileNotFoundError` for images | Check that `eyepacs_images_dir` points to the folder with actual `.jpeg` files |
| Internet is off | Go to Settings -> Internet -> toggle On (needed for git clone) |